In [ ]:

# ============================================
# RGM peak deviator stress prediction
# IBM ML0101EN practical exercise
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    LeaveOneOut,
    cross_val_predict
)
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error
)

# 1. Approximate RGM experimental data
# Digitised from published figures

data = {
    'Rubber_Content (%)': [0, 10, 20, 30, 40] * 3,
    'Confining_Pressure (kPa)': [10]*5 + [40]*5 + [70]*5,
    'Peak_Deviator_Stress (kPa)': [
        155, 135, 115, 82, 75,
        350, 295, 255, 250, 220,
        525, 410, 355, 350, 315
    ]
}

df = pd.DataFrame(data)

print("Original RGM dataset:")
print(df)

# 2. Select input features and target
X = df[[
    'Rubber_Content (%)',
    'Confining_Pressure (kPa)'
]]

y = df['Peak_Deviator_Stress (kPa)']

# 3. Define random forest regression model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=4,
    random_state=42
)

# 4. Evaluate using leave-one-out cross-validation
cv = LeaveOneOut()

y_cv = cross_val_predict(
    model,
    X,
    y,
    cv=cv
)

# 5. Calculate evaluation metrics
rmse = np.sqrt(mean_squared_error(y, y_cv))
mae = mean_absolute_error(y, y_cv)
r2 = r2_score(y, y_cv)

print("\nRandom forest regression performance:")
print(f"RMSE: {rmse:.2f} kPa")
print(f"MAE: {mae:.2f} kPa")
print(f"R2 score: {r2:.3f}")

# 6. Train final model using all data
model.fit(X, y)

# 7. Predict peak deviator stress
df['Predicted_Stress (kPa)'] = model.predict(X)
df['CV_Predicted_Stress (kPa)'] = y_cv

print("\nPrediction results:")
print(df)

# 8. Create prediction heatmap dataset
heatmap_data = df.pivot(
    index='Confining_Pressure (kPa)',
    columns='Rubber_Content (%)',
    values='Predicted_Stress (kPa)'
)

# 9. Plot random forest prediction heatmap
sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(9, 6))

sns.heatmap(
    heatmap_data.iloc[::-1],
    annot=True,
    fmt=".0f",
    cmap="coolwarm",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        'label': 'Predicted peak deviator stress (kPa)'
    },
    ax=ax
)

ax.set_xlabel('Rubber Content (%)')
ax.set_ylabel('Confining Pressure (kPa)')
ax.set_title(
    'Random Forest Prediction of RGM Peak Deviator Stress'
)

plt.tight_layout()

# 10. Save and display heatmap
plt.savefig(
    'RGM_ML_Heatmap.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 11. Export prediction results
df.to_csv(
    'RGM_ML_Prediction_Results.csv',
    index=False
)

# 12. Export model evaluation results
evaluation = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R2'],
    'Value': [rmse, mae, r2]
})

evaluation.to_csv(
    'RGM_ML_Evaluation.csv',
    index=False
)

print("\nModel evaluation:")
print(evaluation)

print("\nMachine learning analysis completed.")
